# Agents SDK Style Market Research (Top 3 Sources)

Compact Deepnote-ready workflow using `openai`, `openai-agents`, and `requests` only.


In [6]:
from __future__ import annotations

import asyncio
import json
import os
from concurrent.futures import ThreadPoolExecutor
from typing import Any

import requests
from openai import AsyncOpenAI
from agents import set_default_openai_client
set_default_openai_client(AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"]))

from agents import Agent, RunConfig, Runner, function_tool, set_default_openai_client

INITIAL_TASK = (
    "Research current trends in AI agent tools used by SMB marketing teams. "
    "Focus on recurring use cases, positioning, and common feature patterns."
)
OLOSTEP_BASE_URL = os.getenv("OLOSTEP_BASE_URL", "https://api.olostep.com").rstrip("/")
BRIEF_PATH = "agents_sdk_style_market_research_top3_brief.md"
MODEL_NAME = "gpt-5.2"

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is missing.")
if not os.getenv("OLOSTEP_API_KEY"):
    raise RuntimeError("OLOSTEP_API_KEY is missing.")

set_default_openai_client(AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"]))

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {os.environ['OLOSTEP_API_KEY']}",
    "Content-Type": "application/json",
})


In [3]:
def _request_olostep(path: str, payload: dict[str, Any]) -> dict[str, Any]:
    response = session.post(f"{OLOSTEP_BASE_URL}{path}", json=payload, timeout=60)
    response.raise_for_status()
    data = response.json()
    if not isinstance(data, dict):
        raise ValueError(f"Unexpected Olostep response type: {type(data)}")
    return data

@function_tool
def olostep_answer_tool(task: str) -> dict[str, Any]:
    """Call Olostep Answers and return raw payload."""
    return _request_olostep("/v1/answers", {"task": task})

@function_tool
def olostep_scrape_tool(url: str) -> dict[str, Any]:
    """Call Olostep Scrapes for one URL."""
    return _request_olostep("/v1/scrapes", {"url_to_scrape": url, "formats": ["markdown", "text"]})

def parse_json_object(value: Any) -> dict[str, Any]:
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        text = value.strip()
        if text.startswith("```"):
            text = text.strip("`").replace("json", "", 1).strip()
        try:
            parsed = json.loads(text)
            return parsed if isinstance(parsed, dict) else {}
        except json.JSONDecodeError:
            return {}
    return {}

def unique_http_urls(items: list[Any]) -> list[str]:
    out, seen = [], set()
    for item in items:
        candidate = str(item).strip()
        if not candidate.startswith(("http://", "https://")):
            continue
        if candidate in seen:
            continue
        seen.add(candidate)
        out.append(candidate)
    return out

def run_async(coro):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coro).result()

def compact_text(value: Any, limit: int = 6000) -> str:
    text = str(value or "").strip()
    return text[:limit]


In [4]:
research_agent = Agent(
    name="research_agent",
    model=MODEL_NAME,
    tools=[olostep_answer_tool, olostep_scrape_tool],
    instructions=(
        "Always keep INITIAL_TASK central.\n"
        "Run this exact flow:\n"
        "1) Call olostep_answer_tool once with INITIAL_TASK.\n"
        "2) Parse result.json_content and result.sources.\n"
        "3) Select top 3 unique URLs (prefer result.sources order).\n"
        "4) Scrape those top 3 URLs with olostep_scrape_tool.\n"
        "Return strict JSON only with keys: initial_task, answer_summary, answer_json_content, answer_sources, top3_sources, scraped_pages."
    ),
)

extraction_agent = Agent(
    name="extraction_agent",
    model=MODEL_NAME,
    instructions=(
        "Always include INITIAL_TASK context.\n"
        "Extract concrete market signals from provided summary + scraped context only.\n"
        "Return strict JSON with: signals (list of objects).\n"
        "Each signal object: topic, use_case, positioning_pattern, feature_pattern, evidence, source_url."
    ),
)

trend_agent = Agent(
    name="trend_agent",
    model=MODEL_NAME,
    instructions=(
        "Always include INITIAL_TASK context.\n"
        "Analyze recurring patterns from extracted signals.\n"
        "Return strict JSON with: trends (list) and summary (string).\n"
        "Each trend object: trend, why_now, supporting_signals, source_urls, confidence_0_to_1."
    ),
)

brief_agent = Agent(
    name="brief_agent",
    model=MODEL_NAME,
    instructions=(
        "Always include INITIAL_TASK context.\n"
        "Write a concise technical research brief in markdown.\n"
        "Use sections: Executive Summary, Top Trends, Recurring Use Cases, Positioning Patterns, Feature Patterns, Sources.\n"
        "Be specific and evidence-linked, but compact."
    ),
)


In [7]:
research_prompt = f"""
INITIAL_TASK:
{INITIAL_TASK}

Use tools to complete the flow exactly and return strict JSON only.
"""

research_run = run_async(Runner.run(
    research_agent,
    input=research_prompt,
    run_config=RunConfig(model=MODEL_NAME),
))
research_payload = parse_json_object(research_run.final_output)

if not research_payload.get("top3_sources"):
    # Deterministic fallback if tool orchestration output is incomplete.
    raw_answer = olostep_answer_tool(INITIAL_TASK)
    result = raw_answer.get("result", {}) if isinstance(raw_answer.get("result"), dict) else {}
    json_content = parse_json_object(result.get("json_content"))
    answer_sources = []
    for item in result.get("sources", []):
        if isinstance(item, str):
            answer_sources.append(item)
        elif isinstance(item, dict) and isinstance(item.get("url"), str):
            answer_sources.append(item["url"])
    json_urls = json_content.get("urls", []) if isinstance(json_content.get("urls", []), list) else []
    top3 = unique_http_urls(answer_sources + json_urls)[:3]

    scraped_pages = []
    for url in top3:
        raw_scrape = olostep_scrape_tool(url)
        scrape_result = raw_scrape.get("result", {}) if isinstance(raw_scrape.get("result"), dict) else {}
        scraped_pages.append({
            "url": url,
            "title": scrape_result.get("metadata", {}).get("title", ""),
            "content": compact_text(scrape_result.get("markdown_content") or scrape_result.get("text_content")),
        })

    research_payload = {
        "initial_task": INITIAL_TASK,
        "answer_summary": result.get("summary", ""),
        "answer_json_content": json_content,
        "answer_sources": unique_http_urls(answer_sources),
        "top3_sources": top3,
        "scraped_pages": scraped_pages,
    }

print("Top 3 sources selected:")
print(json.dumps(research_payload.get("top3_sources", []), indent=2))
print("\nResearch payload preview:")
print(json.dumps({
    "initial_task": research_payload.get("initial_task"),
    "answer_sources_count": len(research_payload.get("answer_sources", [])),
    "top3_sources": research_payload.get("top3_sources", []),
}, indent=2))


Top 3 sources selected:
[
  "https://www.cigen.io/insights/ai-in-smb-10-practical-use-cases-you-can-start-with-today",
  "https://bertomill.medium.com/how-ai-agents-are-leveling-the-playing-field-for-small-businesses-in-2026-44dfdc45a178",
  "https://www.digitalapplied.com/blog/ai-agent-workflows-smb-revenue-streams-guide"
]

Research payload preview:
{
  "initial_task": "Research current trends in AI agent tools used by SMB marketing teams. Focus on recurring use cases, positioning, and common feature patterns.",
  "answer_sources_count": 27,
  "top3_sources": [
    "https://www.cigen.io/insights/ai-in-smb-10-practical-use-cases-you-can-start-with-today",
    "https://bertomill.medium.com/how-ai-agents-are-leveling-the-playing-field-for-small-businesses-in-2026-44dfdc45a178",
    "https://www.digitalapplied.com/blog/ai-agent-workflows-smb-revenue-streams-guide"
  ]
}


In [8]:
extraction_prompt = f"""
INITIAL_TASK:
{INITIAL_TASK}

Extract signals from this research package. Return strict JSON only.

RESEARCH_PACKAGE:
{json.dumps(research_payload, ensure_ascii=False)}
"""

extraction_run = run_async(Runner.run(
    extraction_agent,
    input=extraction_prompt,
    run_config=RunConfig(model=MODEL_NAME),
))
extraction_payload = parse_json_object(extraction_run.final_output)
signals = extraction_payload.get("signals", []) if isinstance(extraction_payload.get("signals"), list) else []

print("Signals extracted:", len(signals))
print(json.dumps(signals[:5], indent=2, ensure_ascii=False))


Signals extracted: 14
[
  {
    "topic": "No-code/low-code accessibility for SMB AI marketing automation",
    "use_case": "SMB teams adopt AI for marketing-adjacent workflows via no-code/low-code and cloud tooling rather than custom engineering",
    "positioning_pattern": "“Accessible to SMBs” framing; emphasizes ease of adoption through no-code/low-code + cloud services",
    "feature_pattern": "Workflow automation building blocks; cloud-delivered AI capabilities; personalization as a core value prop",
    "evidence": "“AI is accessible to SMBs via no-code/low-code and cloud services; highlights workflow automation and personalization as core value.”",
    "source_url": "https://www.cigen.io/insights/ai-in-smb-10-practical-use-cases-you-can-start-with-today"
  },
  {
    "topic": "LLM drafting/content assistance as a practical SMB marketing use case",
    "use_case": "AI-powered drafting for marketing communications/content production",
    "positioning_pattern": "“Practical use cas

In [9]:
trend_prompt = f"""
INITIAL_TASK:
{INITIAL_TASK}

Run trend analysis from the extracted signals. Return strict JSON only.

SIGNALS_JSON:
{json.dumps(signals, ensure_ascii=False)}
"""

trend_run = run_async(Runner.run(
    trend_agent,
    input=trend_prompt,
    run_config=RunConfig(model=MODEL_NAME),
))
trend_payload = parse_json_object(trend_run.final_output)
trends = trend_payload.get("trends", []) if isinstance(trend_payload.get("trends"), list) else []

print("Trends identified:", len(trends))
print(json.dumps(trends[:5], indent=2, ensure_ascii=False))


Trends identified: 9
[
  {
    "trend": "No-code/low-code + cloud delivery as the default adoption path for SMB marketing agents",
    "why_now": "SMB marketing teams want AI benefits without hiring engineers; mature cloud AI and no-code automation platforms make agent-style workflows deployable by non-technical operators.",
    "supporting_signals": [
      "“AI is accessible to SMBs via no-code/low-code and cloud services; highlights workflow automation and personalization as core value.”",
      "“Emphasizes ‘productized’ AI agent workflows built on automation platforms (Make, n8n, Zapier)… deploy in hours and run autonomously.”"
    ],
    "source_urls": [
      "https://www.cigen.io/insights/ai-in-smb-10-practical-use-cases-you-can-start-with-today",
      "https://www.digitalapplied.com/blog/ai-agent-workflows-smb-revenue-streams-guide"
    ],
    "confidence_0_to_1": 0.84
  },
  {
    "trend": "Productized, templatized agent workflows (repeatable playbooks) instead of bespoke “A

In [10]:
brief_prompt = f"""
INITIAL_TASK:
{INITIAL_TASK}

Generate the final concise technical research brief in markdown.

ANSWER_SUMMARY_AND_CONTEXT:
{json.dumps({
    'answer_summary': research_payload.get('answer_summary', ''),
    'top3_sources': research_payload.get('top3_sources', []),
    'scraped_pages': research_payload.get('scraped_pages', []),
}, ensure_ascii=False)}

EXTRACTED_SIGNALS:
{json.dumps(signals, ensure_ascii=False)}

TREND_ANALYSIS:
{json.dumps(trends, ensure_ascii=False)}
"""

brief_run = run_async(Runner.run(
    brief_agent,
    input=brief_prompt,
    run_config=RunConfig(model=MODEL_NAME),
))
final_brief = str(brief_run.final_output).strip()

with open(BRIEF_PATH, "w", encoding="utf-8") as f:
    f.write(final_brief + "\n")

print(f"Saved brief to: {BRIEF_PATH}")
print("\n--- Brief Preview ---\n")
print(final_brief[:3000])


Saved brief to: agents_sdk_style_market_research_top3_brief.md

--- Brief Preview ---

## Executive Summary (INITIAL_TASK)
SMB marketing teams are adopting **AI agent tools** that move beyond single-task “AI copy” into **agentic workflows**: multi-step, integration-driven automations that can **perceive context, decide, and take actions** across common SMB systems (CRM, forms, calendars, review platforms). The dominant direction is toward **productized, template-based playbooks** deployed via **no-code/low-code automation platforms** (e.g., Zapier/Make/n8n) with **human-in-the-loop routing/escalation** for exceptions and brand safety. Evidence points to strongest clustering around **lead handling/conversion ops**, **reputation management**, and **content + nurture execution**.  
Sources: Cigen (SMB AI practical use cases)【https://www.cigen.io/insights/ai-in-smb-10-practical-use-cases-you-can-start-with-today】; DigitalApplied (productized agent workflows on Make/n8n/Zapier)【https://www.